In [57]:
import pandas as pd
from pathlib import Path

raw_dir = Path("../data/raw")

files = {
    "salaries": "american_soccer_analysis_mls_salaries_players.csv",
    "goals_added_gk": "american_soccer_analysis_mls_goals-added_goalkeepers.csv",
    "goals_added": "american_soccer_analysis_mls_goals-added_players.csv",
    "xgoals": "american_soccer_analysis_mls_xgoals_players.csv",
    "xpass": "american_soccer_analysis_mls_xpass_players.csv",
    "uslc_xgoals": "american_soccer_analysis_uslc_xgoals_players.csv",
    "uslc_xpass": "american_soccer_analysis_uslc_xpass_players.csv",
    "uslc_goals_added": "american_soccer_analysis_uslc_goals-added_players.csv",
    "uslc_goals_added_gk": "american_soccer_analysis_uslc_goals-added_goalkeepers.csv",
}

asa_tables = {
    source: pd.read_csv(raw_dir / filename)
    for source, filename in files.items()
}

for source, df in asa_tables.items():
    print(f"\n--- {source} ---")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")

    if "Season" in df.columns:
        print(f"Seasons: {sorted(df['Season'].dropna().unique().tolist())}")



--- salaries ---
Shape: (15490, 8)
Columns: ['Unnamed: 0', 'Player', 'Team', 'Season', 'Position', 'Base Salary', 'Guaranteed Compensation', 'Date']
Seasons: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

--- goals_added_gk ---
Shape: (572, 12)
Columns: ['Unnamed: 0', 'Player', 'Team', 'Season', 'Minutes', 'Claiming', 'Fielding', 'Handling', 'Passing', 'Shotstopping', 'Sweeping', 'Goals Added']
Seasons: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

--- goals_added ---
Shape: (6399, 13)
Columns: ['Unnamed: 0', 'Player', 'Team', 'Season', 'Position', 'Minutes', 'Dribbling', 'Fouling', 'Interrupting', 'Passing', 'Receiving', 'Shooting', 'Goals Added']
Seasons: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

--- xgoals ---
Shape: (6970, 20)
Columns: ['Unnamed: 0', 'Player', 'Team', 'Season', 'Position', 'Minutes', 'Shots', 'SoT', 'G', 'xG', 'xPlace', 'G-xG', 'KeyP', 'A', 'xA', 'A-xA', 'G+A', 'xG+xA', 'PA', 'xPA']
Seasons: [2016, 2017, 201

In [58]:
keys = ["Player", "Season"]
results = []

for source, df in asa_tables.items():
    historical = df[df["Season"].between(2016, 2025)]

    results.append({
        "source": source,
        "rows": len(historical),
        "unique_player_seasons": (
            historical[keys].drop_duplicates().shape[0]
        ),
        "repeated_player_season_rows": (
            historical.duplicated(subset=keys).sum()
        ),
        "missing_player": historical["Player"].isna().sum(),
        "missing_season": df["Season"].isna().sum(),
    })

coverage_structure = pd.DataFrame(results)

display(coverage_structure)


,source,rows,unique_player_seasons,repeated_player_season_rows,missing_player,missing_season
0,salaries,14579,8196,6383,0,0
1,goals_added_gk,572,572,0,0,0
2,goals_added,6399,6397,2,0,0
3,xgoals,6970,6968,2,0,0
4,xpass,6970,6968,2,0,0
5,uslc_xgoals,7282,7278,4,0,0
6,uslc_xpass,7282,7278,4,0,0
7,uslc_goals_added,6587,6583,4,0,0
8,uslc_goals_added_gk,695,695,0,0,0


In [59]:
salary_population = (
    asa_tables["salaries"]
    .loc[
        lambda df: df["Season"].between(2016, 2025),
        keys
    ]
    .drop_duplicates()
)

mls_performance_population = pd.concat(
    [
        df.loc[
            df["Season"].between(2016, 2025),
            keys
        ]
        for source, df in asa_tables.items()
        if source in {
            "goals_added_gk",
            "goals_added",
            "xgoals",
            "xpass",
        }
    ],
    ignore_index=True
).drop_duplicates()

salary_coverage = salary_population.merge(
    mls_performance_population.assign(has_mls_performance=True),
    on=keys,
    how="left",
    validate="one_to_one"
)

salary_only = (
    salary_coverage[
        salary_coverage["has_mls_performance"].isna()
    ]
    .drop(columns="has_mls_performance")
    .sort_values(["Season", "Player"])
    .reset_index(drop=True)
)

mls_matched = salary_coverage["has_mls_performance"].notna().sum()

print(f"Salary player-seasons: {len(salary_population):,}")
print(f"MLS performance player-seasons: {len(mls_performance_population):,}")
print(f"MLS-matched salary player-seasons: {mls_matched:,}")
print(f"MLS-unmatched salary player-seasons: {len(salary_only):,}")
print(f"MLS coverage: {mls_matched / len(salary_population):.2%}")

Salary player-seasons: 8,196
MLS performance player-seasons: 6,968
MLS-matched salary player-seasons: 6,847
MLS-unmatched salary player-seasons: 1,349
MLS coverage: 83.54%


In [60]:
# Unique player-seasons across USLC performance sources
uslc_sources = [
    "uslc_xgoals",
    "uslc_xpass",
    "uslc_goals_added",
    "uslc_goals_added_gk",
]

uslc_population = pd.concat(
    [
        asa_tables[source].loc[
            lambda df: df["Season"].between(2016, 2025),
            keys
        ]
        for source in uslc_sources
    ],
    ignore_index=True
).drop_duplicates()

print(f"USLC performance player-seasons: {len(uslc_population):,}")

USLC performance player-seasons: 7,278


In [61]:
# Compare MLS-unmatched salary population against USLC
coverage_check = salary_only.merge(
    uslc_population.assign(uslc_match=True),
    on=keys,
    how="left",
    validate="one_to_one"
)

coverage_check["uslc_match"] = (
    coverage_check["uslc_match"].notna()
)

coverage_check["uslc_coverage"] = "no_exact_match"

coverage_check.loc[
    coverage_check["uslc_match"],
    "uslc_coverage"
] = "exact_name_season_match"

# USLC data starts in 2017
coverage_check.loc[
    coverage_check["Season"] == 2016,
    "uslc_coverage"
] = "season_unavailable"

assert len(coverage_check) == len(salary_only)

print("MLS-unmatched salary player-seasons by USLC coverage:")

display(
    coverage_check["uslc_coverage"]
    .value_counts()
    .rename_axis("uslc_coverage")
    .reset_index(name="player_seasons")
)

MLS-unmatched salary player-seasons by USLC coverage:


,uslc_coverage,player_seasons
0,no_exact_match,819
1,exact_name_season_match,432
2,season_unavailable,98


In [62]:
# Consolidate coverage classifications
coverage_summary = pd.DataFrame({
    "classification": [
        "mls_exact_match",
        "uslc_exact_match_only",
        "no_exact_match_either",
        "mls_unmatched_uslc_unavailable",
    ],
    "player_seasons": [
        mls_matched,
        (coverage_check["uslc_coverage"] == "exact_name_season_match").sum(),
        (coverage_check["uslc_coverage"] == "no_exact_match").sum(),
        (coverage_check["uslc_coverage"] == "season_unavailable").sum(),
    ],
})

coverage_summary["percentage"] = (
    coverage_summary["player_seasons"]
    / len(salary_population) * 100
).round(2)

assert coverage_summary["player_seasons"].sum() == len(salary_population)
assert not salary_coverage.duplicated(subset=keys).any()
assert not coverage_check.duplicated(subset=keys).any()

display(coverage_summary)

,classification,player_seasons,percentage
0,mls_exact_match,6847,83.54
1,uslc_exact_match_only,432,5.27
2,no_exact_match_either,819,9.99
3,mls_unmatched_uslc_unavailable,98,1.20


## Coverage audit — Findings and implications

Before attempting to explain player value through performance, I need to understand how much of the salary population is represented in the performance data.

Between 2016 and 2025, the salary dataset contains 8,196 unique name-season combinations. Of these, 83.54% have a corresponding MLS performance record. But what about the remaining 1,349? Could some of these players have been playing in USLC while under contract with an MLS club?

| Coverage | Player-seasons | Share |
|---|---:|---:|
| MLS match | 6,847 | 83.54% |
| USLC match without MLS match | 432 | 5.27% |
| No match in either competition | 819 | 9.99% |
| No MLS match; USLC unavailable | 98 | 1.20% |

The USLC comparison accounts for another 432 name-season combinations, although the absence of USLC data for 2016 limits the analysis.

Two considerations inform the next stage. First, coverage is currently established through names rather than verified player identifiers. Second, the MLSPA publishes salary figures at different points during the season, meaning repeated salary observations must be interpreted in the context of its reporting cycle.

The next step is to establish player identities, integrate the salary and performance tables, and determine how to represent seasonal compensation and treat missing performance observations before defining the modeling population.